# LC 11 — Container With Most Water
**Day 52 | Two Pointers Review | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Water is limited by the shorter wall.
If you move the taller pointer inward you can only lose area, so
always move the <em>shorter</em> side — it's the only move that
could possibly increase the water level.
</div>

## Official Problem Statement

You are given an integer array `height` of length `n`. There are
`n` vertical lines drawn such that the two endpoints of the i-th
line are `(i, 0)` and `(i, height[i])`.

Find two lines that together with the x-axis form a container,
such that the container contains the **most water**.

Return the **maximum amount of water** a container can store.

Notice that you may not slant the container.

**Constraints:**
- `n == height.length`
- `2 <= n <= 10^5`
- `0 <= height[i] <= 10^4`

## What This Is Actually Asking

Pick two indices i < j; the water they can hold is
`min(height[i], height[j]) * (j - i)`.
We want to maximise that product over all pairs.
Brute force checks every pair in O(n²); we need O(n).
The key observation: the bottleneck is always the shorter bar,
so moving the taller bar inward can never help — we always move
the shorter bar hoping to find a taller replacement.

## Walk Through an Example by Hand

```
height = [1, 8, 6, 2, 5, 4, 8, 3, 7]
          0  1  2  3  4  5  6  7  8

left=0, right=8: water=min(1,7)*(8-0)=1*8=8    → move left (h=1 shorter)
left=1, right=8: water=min(8,7)*(8-1)=7*7=49   → move right (h=7 shorter)
left=1, right=7: water=min(8,3)*(7-1)=3*6=18   → move right
left=1, right=6: water=min(8,8)*(6-1)=8*5=40   → move either (tie: right)
left=1, right=5: water=min(8,4)*(5-1)=4*4=16   → move right
left=1, right=4: water=min(8,5)*(4-1)=5*3=15   → move right
left=1, right=3: water=min(8,2)*(3-1)=2*2=4    → move right
left=1, right=2: water=min(8,6)*(2-1)=6*1=6    → move right
left=1, right=1: stop

max = 49  ✓
```

## The Picture

```
  height: [1, 8, 6, 2, 5, 4, 8, 3, 7]
           0  1  2  3  4  5  6  7  8

  8 |  █              █
  7 |  █              █        █
  6 |  █  █           █
  5 |  █  █     █     █
  4 |  █  █     █  █  █
  3 |  █  █     █  █  █  █
  2 |  █  █  █  █  █  █  █
  1 |█ █  █  █  █  █  █  █  █
     ─────────────────────────
     L=1              R=8
     ←─────── 7 ──────────────►
     water = min(8,7) * 7 = 49

  Rule: always move the SHORTER pointer inward.
```

## When To Use This Pattern

- When you need to **maximise or minimise a product** of a
  width and a height term over pairs, think two pointers.
- When moving one pointer can only hurt, think two pointers
  (**greedy elimination** of dominated choices).
- When a brute-force O(n²) pair search feels natural, check
  if the problem has a monotone property that lets you skip
  provably suboptimal pairs.
- The same "move the limiting side" logic applies to Trapping
  Rain Water (LC 42) — a direct extension of this pattern.
- When the array is NOT sorted but two-end scanning is still
  valid due to a min/max bottleneck, think this pattern.

## The Approach

Start with the widest possible container (left=0, right=n-1).
Compute the water for the current pair and update the maximum.
Move whichever pointer points to the shorter bar inward —
moving the taller one inward only reduces both width and the
effective height, so it can't improve the result.
Repeat until the pointers meet; the tracked maximum is the answer.

In [2]:
from typing import List

In [3]:
def test_harness(func):
    cases = [
        ([1, 8, 6, 2, 5, 4, 8, 3, 7], 49),
        ([1, 1],                        1),
        ([4, 3, 2, 1, 4],              16),
        ([1, 2, 1],                     2),
        ([2, 3, 4, 5, 18, 17, 6],      17),
    ]
    passed = 0
    for height, expected in cases:
        result = func(height[:])
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"{status} | height={height}"
            f" | expected={expected}, got={result}"
        )
    print(f"\n{passed}/{len(cases)} tests passed.")

In [5]:
def max_area(height: List[int]) -> int:
    """
    Return the maximum water a container formed by two bars
    in `height` can hold.

    Strategy: converging two pointers.
      water = min(height[l], height[r]) * (r - l)
      Move the shorter pointer inward each step.

    Time : O(n)   — each pointer moves at most n times
    Space: O(1)

    Args:
        height: list of non-negative bar heights

    Returns:
        Maximum water volume (integer).
    """
    l, r = 0, len(height) - 1
    res = 0
    while l<r:
        res = max(res, (r-l) * min (height[r], height[l]) )
        if height[l] < height[r]:
            l+=1
        else:
            r-=1
    return res

print(max_area([1,8,6,2,5,4,8,3,7]))  # 49
print(max_area([1,1]))                 # 1
print(max_area([4,3,2,1,4]))           # 16
print(max_area([1,0,0,0,1]))           # 4
test_harness(max_area)        
        


49
1
16
4
PASSED | height=[1, 8, 6, 2, 5, 4, 8, 3, 7] | expected=49, got=49
PASSED | height=[1, 1] | expected=1, got=1
PASSED | height=[4, 3, 2, 1, 4] | expected=16, got=16
PASSED | height=[1, 2, 1] | expected=2, got=2
PASSED | height=[2, 3, 4, 5, 18, 17, 6] | expected=17, got=17

5/5 tests passed.


In [ ]:
# Uncomment and run when solution is ready
# test_harness(max_area)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (all pairs) | O(n²) | O(1) | TLE on large n |
| Two pointers (optimal) | O(n) | O(1) | Single pass |

## Real World Connection

In **capacity planning at AWS**, you might model available
bandwidth between data-centre racks as bar heights — the
bottleneck link determines actual throughput, mirroring the
`min(height)` constraint.
Finding the pair of racks that maximises data transfer without
scanning every combination is exactly this algorithm.
At **Citi**, portfolio stress tests sometimes need to find two
offsetting hedges that maximise net protection; the two-pointer
greedy principle (skip dominated options) reduces search space
from O(n²) to O(n) when positions are ranked by a monotone key.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra